In [ ]:
# RT-DETR Training, Inference & Evaluation
Real-Time DEtection TRansformer - Vision Transformer based object detector.  
Fine-tuning on Military Assets Dataset (13 classes).

In [ ]:
## 1. Setup & Dependencies

In [ ]:
%pip install ultralytics kagglehub -q

In [ ]:
import kagglehub
from pathlib import Path

DATASET_DIR = Path(kagglehub.dataset_download('rawsi18/military-assets-dataset-12-classes-yolo8-format'))
print(f"Dataset path: {DATASET_DIR}")

DATASET_YAML = DATASET_DIR / "military_object_dataset" / "military_dataset.yaml"
print(f"YAML config: {DATASET_YAML}")

## 2. Training
RT-DETR-L (Large) - pretrained on COCO, fine-tuned on military assets.

In [ ]:
from ultralytics import RTDETR

# Load pretrained RT-DETR-L (Vision Transformer backbone)
model = RTDETR("rtdetr-l.pt")

results = model.train(
    data=str(DATASET_YAML),
    epochs=100,
    imgsz=640,
    batch=8,  # mem intensive choose batch based on GPU 
    device=0,
    project="runs/rtdetr",
    name="rtdetr_military",
    save=True,
    save_period=10,
    patience=20,  # early stopping
    exist_ok=True
)

## 3. Inference
Load trained checkpoint and run predictions on test images.

In [ ]:
from ultralytics import RTDETR
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np

# trained checkpoint 
CHECKPOINT_PATH = r""

model = RTDETR(CHECKPOINT_PATH)
print(f"Model loaded from: {CHECKPOINT_PATH}")
print(f"Classes: {model.names}")

In [ ]:
test_images = list(DATASET_DIR.glob("**/test/images/*.jpg"))
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    save=True,
    save_txt=True,
    project="runs/rtdetr_inference",
    name="military_test",
    exist_ok=True
)

print("Inference complete!")

In [ ]:
def display_results(results, max_images=4):
    n_images = min(len(results), max_images)
    cols = 2
    rows = (n_images + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(14, 7 * rows))
    axes = axes.flatten() if n_images > 1 else [axes]
    
    for idx, result in enumerate(results[:max_images]):
        annotated = result.plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        axes[idx].axis('off')
        
        n_detections = len(result.boxes)
        classes_detected = [model.names[int(c)] for c in result.boxes.cls] if n_detections > 0 else []
        axes[idx].set_title(f"Detections: {n_detections}\n{', '.join(set(classes_detected))[:50]}", fontsize=10)
    
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

display_results(results, max_images=4)

In [ ]:
def analyze_detections(results, model):
    total_detections = 0
    class_counts = {}
    confidence_scores = []
    
    for result in results:
        boxes = result.boxes
        total_detections += len(boxes)
        
        for box in boxes:
            cls_id = int(box.cls)
            cls_name = model.names[cls_id]
            conf = float(box.conf)
            class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            confidence_scores.append(conf)
    
    print("=" * 50)
    print("INFERENCE SUMMARY")
    print("=" * 50)
    print(f"Total images processed: {len(results)}")
    print(f"Total detections: {total_detections}")
    print(f"Average detections per image: {total_detections / len(results):.2f}")
    
    if confidence_scores:
        print(f"\nConfidence Statistics:")
        print(f"  Mean: {np.mean(confidence_scores):.3f}")
        print(f"  Min:  {np.min(confidence_scores):.3f}")
        print(f"  Max:  {np.max(confidence_scores):.3f}")
    
    if class_counts:
        print(f"\nDetections by class:")
        for cls_name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
            print(f"  {cls_name}: {count}")
    
    return class_counts, confidence_scores

class_counts, confidence_scores = analyze_detections(results, model)

In [ ]:
def infer_single_image(model, image_path, conf=0.25, show=True):
    result = model.predict(source=image_path, conf=conf, iou=0.45, imgsz=640, verbose=False)[0]
    
    if show:
        annotated = result.plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_rgb)
        plt.axis('off')
        plt.title(f"Detections: {len(result.boxes)}")
        plt.show()
    
    detections = []
    for box in result.boxes:
        detections.append({
            'class': model.names[int(box.cls)],
            'confidence': float(box.conf),
            'bbox_xyxy': box.xyxy[0].tolist(),
            'bbox_xywh': box.xywh[0].tolist()
        })
    return detections

if test_images:
    detections = infer_single_image(model, test_images[0])
    print("\nDetection details:")
    for det in detections:
        print(f"  {det['class']}: {det['confidence']:.2f} @ {[int(x) for x in det['bbox_xyxy']]}")

## 4. Evaluation
Compute mAP metrics on the test set.

In [ ]:
metrics = model.val(
    data=str(DATASET_YAML),
    imgsz=640,
    batch=8,
    conf=0.001,
    iou=0.6,
    split="test",
    project="runs/rtdetr_val",
    name="military_eval",
    exist_ok=True
)

print("\n" + "=" * 50)
print("EVALUATION METRICS (Military Assets Test Set)")
print("=" * 50)
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

In [ ]:
# Per-class AP breakdown
print("\n" + "=" * 50)
print("PER-CLASS AP@50")
print("=" * 50)
for i, ap in enumerate(metrics.box.ap50):
    print(f"  {model.names[i]:20s}: {ap:.4f}")

## 5. Deployment 
Export model for deployment.

In [ ]:
# model.export(format="onnx", imgsz=640, simplify=True)
# model.export(format="engine", imgsz=640, half=True)  # TensorRT